# Civic — Garbage YOLOv8m
Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
!pip -q install ultralytics roboflow huggingface_hub

In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')
os.environ['HF_TOKEN'] = getpass.getpass('HF write token (skip to not upload): ')

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
trash = rf.workspace('fyp-bfx3h').project('yolov8-trash-detections').version(4).download('yolov8', location='datasets/garbage')
print('Garbage dataset:', trash.location)

In [ ]:
from ultralytics import YOLO
YOLO('yolov8m.pt').train(
    data=f'{trash.location}/data.yaml',
    epochs=150, imgsz=640, name='garbage_v2',
    patience=30, batch=-1, cache='ram', plots=True, exist_ok=True,
)

In [ ]:
import os
PT = 'runs/detect/garbage_v2/weights/best.pt'
print(PT, '->', os.path.getsize(PT) // 1024, 'KB')
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    api.upload_file(path_or_fileobj=PT, path_in_repo='weights/garbage_v2.pt', repo_id='arjun-vegeta/civic-yolo', repo_type='space')
    api.restart_space(repo_id='arjun-vegeta/civic-yolo')
    print('uploaded + Space restarted')
else:
    print('No HF_TOKEN — download best.pt from the Files panel manually.')